In [1]:
import os
import pynvml
import torch

def select_gpu_interactively():
    # Initialize NVML
    pynvml.nvmlInit()
    device_count = pynvml.nvmlDeviceGetCount()
    print(f"Detected {device_count} GPU(s):\n")

    for i in range(device_count):
        handle = pynvml.nvmlDeviceGetHandleByIndex(i)
        mem = pynvml.nvmlDeviceGetMemoryInfo(handle)
        util = pynvml.nvmlDeviceGetUtilizationRates(handle)
        name = pynvml.nvmlDeviceGetName(handle)
        if isinstance(name, bytes):
            name = name.decode('utf-8')
        print(f"GPU {i}: {name}")
        print(f"  Memory Usage: {mem.used / 1024 ** 2:.1f} MB / {mem.total / 1024 ** 2:.1f} MB")
        print(f"  GPU Utilization: {util.gpu}%, Memory Utilization: {util.memory}%\n")

    # Prompt for manual selection
    selected = input("Enter the physical GPU index you want to use (e.g., 0 / 1 / 2 / 3): ").strip()
    if not selected.isdigit() or int(selected) >= device_count:
        print("Invalid input. Aborting.")
        pynvml.nvmlShutdown()
        return None

    selected = int(selected)
    os.environ["CUDA_VISIBLE_DEVICES"] = str(selected)  # Limit visibility to selected GPU
    pynvml.nvmlShutdown()

    # Check if PyTorch can see it
    if torch.cuda.is_available():
        device = torch.device("cuda:0")  # Mapped to logical 0 in PyTorch
        name = torch.cuda.get_device_name(device)
        print(f"\n GPU {selected} selected (mapped to cuda:0 in PyTorch)")
        print(f"Device name: {name}")
        return device
    else:
        print(" No available CUDA device detected.")
        return None

device = select_gpu_interactively()

Detected 4 GPU(s):

GPU 0: NVIDIA L40
  Memory Usage: 33990.4 MB / 46068.0 MB
  GPU Utilization: 97%, Memory Utilization: 92%

GPU 1: NVIDIA L40
  Memory Usage: 698.9 MB / 46068.0 MB
  GPU Utilization: 0%, Memory Utilization: 0%

GPU 2: NVIDIA L40
  Memory Usage: 696.2 MB / 46068.0 MB
  GPU Utilization: 0%, Memory Utilization: 0%

GPU 3: NVIDIA L40
  Memory Usage: 9402.2 MB / 46068.0 MB
  GPU Utilization: 0%, Memory Utilization: 0%


 GPU 3 selected (mapped to cuda:0 in PyTorch)
Device name: NVIDIA L40


In [ ]:
import numpy as np
import torch
import torch.nn as nn
import torch.nn.functional as F